# Advanced Models: XGBoost, GRU, Bidirectional LSTM, CNN-LSTM Hybrid

Bank stock price forecasting -- Lloyds, Barclays, Goldman Sachs.

This notebook is a standalone companion to the ARIMA / Random Forest / LSTM notebook: it only contains the four newer models (XGBoost, GRU, Bidirectional LSTM, CNN-LSTM), built on the same cleaned dataset and the same per-bank leakage-safe splits, so results are directly comparable but kept in their own file.


## Import Libraries


In [ ]:
!pip install xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, GRU, Bidirectional, Conv1D, MaxPooling1D, Dense, Dropout
)
from tensorflow.keras.optimizers import Adam

import joblib


## Load Dataset


In [ ]:
df = pd.read_csv('/content/all_bank_stock_data.csv')

df.head()


## Data Preprocessing


In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

numeric_cols = ['Open','High','Low','Close','Volume']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(inplace=True)

encoder = LabelEncoder()
df['Bank'] = encoder.fit_transform(df['Bank'])

df = df.sort_values('Date')


## Feature Engineering


In [ ]:
# Lag features and moving averages, computed PER BANK -- mixing banks together
# before shifting/rolling would leak one bank's price history into another's
# features (the same bug that was fixed in the original Random Forest notebook).
df['Lag1'] = df.groupby('Bank')['Close'].shift(1)
df['Lag2'] = df.groupby('Bank')['Close'].shift(2)
df['Lag3'] = df.groupby('Bank')['Close'].shift(3)

df['MA5'] = df.groupby('Bank')['Close'].transform(lambda s: s.rolling(5).mean())
df['MA10'] = df.groupby('Bank')['Close'].transform(lambda s: s.rolling(10).mean())

df.dropna(inplace=True)


## Train-Test Split


XGBoost is a tabular tree-based model, so it uses the same lag/moving-average
feature table and change-target approach as Random Forest did. GRU, Bidirectional
LSTM, and CNN-LSTM are sequence models, so they get their own per-bank scaled
60-day windows further down, right before the sequence models start.


In [ ]:
X_rf = df[['Bank','Open','High','Low','Volume','Lag1','Lag2','Lag3','MA5','MA10']]

# Predict the day-to-day CHANGE in price (Close - Lag1), not the raw Close level --
# tree-based models can't extrapolate beyond the price range seen during training,
# which badly under-predicts strongly trending stocks. The actual price is
# reconstructed afterwards as Lag1 + predicted change.
y_rf_change = df['Close'] - df['Lag1']
y_rf_close = df['Close']


In [ ]:
X_train_rf, X_test_rf, y_train_change_rf, y_test_change_rf, y_train_close_rf, y_test_close_rf = train_test_split(
    X_rf,
    y_rf_change,
    y_rf_close,
    test_size=0.2,
    shuffle=False,
)


# XGBoost


## Hyperparameter Tuning (XGBoost)


In [ ]:
# TimeSeriesSplit instead of default K-fold -- regular CV would validate on data
# that comes before some of its training data, which is leakage for a time series.
xgb_param_dist = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
}

xgb_tscv = TimeSeriesSplit(n_splits=5)

xgb_search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42, objective="reg:squarederror"),
    param_distributions=xgb_param_dist,
    n_iter=20,
    cv=xgb_tscv,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1,
)

xgb_search.fit(X_train_rf, y_train_change_rf)

print("Best XGBoost params:", xgb_search.best_params_)


## XGBoost Model


In [ ]:
xgb_model = xgb_search.best_estimator_

xgb_model.fit(X_train_rf, y_train_change_rf)

xgb_pred_change = xgb_model.predict(X_test_rf)

# Reconstruct the actual price level from the previous day's close + predicted change.
xgb_pred = X_test_rf['Lag1'].values + xgb_pred_change


## XGBoost Results


In [ ]:
xgb_mae = mean_absolute_error(y_test_close_rf, xgb_pred)

xgb_rmse = np.sqrt(mean_squared_error(y_test_close_rf, xgb_pred))

xgb_r2 = r2_score(y_test_close_rf, xgb_pred)

print("MAE :", xgb_mae)
print("RMSE:", xgb_rmse)
print("R\u00b2  :", xgb_r2)


In [ ]:
plt.figure(figsize=(15,5))

plt.plot(np.array(y_test_close_rf), label='Actual')

plt.plot(xgb_pred, label='XGBoost')

plt.title("XGBoost Prediction")

plt.xlabel("Time")

plt.ylabel("Close Price")

plt.legend()

plt.show()


## Feature Importance (XGBoost)


In [ ]:
xgb_importance = xgb_model.feature_importances_

plt.figure(figsize=(10,6))
plt.bar(X_rf.columns, xgb_importance)
plt.xticks(rotation=45)
plt.title("XGBoost Feature Importance")
plt.show()


## Sequence Data Preparation (GRU / Bidirectional LSTM / CNN-LSTM)


In [ ]:
sequence = 60

# Fit one scaler PER BANK -- Goldman Sachs, Barclays, and Lloyds trade on very
# different price scales, so a single global scaler would distort all three.
# Each scaler is fit ONLY on that bank's TRAINING portion (first 80%, chronologically)
# to avoid leaking the test period's min/max into training.
bank_scalers = {}
bank_train_scaled = {}
bank_test_scaled = {}

for bank_id in sorted(df['Bank'].unique()):
    bank_close = df.loc[df['Bank'] == bank_id].sort_values('Date')[['Close']]

    split_point = int(len(bank_close) * 0.8)
    train_close = bank_close.iloc[:split_point]
    test_close = bank_close.iloc[split_point:]

    scaler = MinMaxScaler()
    scaler.fit(train_close)

    bank_train_scaled[bank_id] = scaler.transform(train_close)
    bank_test_scaled[bank_id] = scaler.transform(test_close)
    bank_scalers[bank_id] = scaler


In [ ]:
bank_sequences_train = {}
bank_sequences_test = {}

for bank_id in bank_train_scaled:
    train_scaled = bank_train_scaled[bank_id]
    test_scaled = bank_test_scaled[bank_id]

    X_tr, y_tr = [], []
    for i in range(sequence, len(train_scaled)):
        X_tr.append(train_scaled[i-sequence:i, 0])
        y_tr.append(train_scaled[i, 0])

    # Test sequences: each uses the most recent `sequence` days of genuinely PAST
    # prices (which may span back into the training period) -- mirrors real
    # deployment and is not leakage.
    full_scaled = np.concatenate([train_scaled, test_scaled])
    X_te, y_te = [], []
    for i in range(len(train_scaled), len(full_scaled)):
        X_te.append(full_scaled[i-sequence:i, 0])
        y_te.append(full_scaled[i, 0])

    bank_sequences_train[bank_id] = (np.array(X_tr), np.array(y_tr))
    bank_sequences_test[bank_id] = (np.array(X_te), np.array(y_te))


In [ ]:
X_train_parts, X_test_parts = [], []
y_train_parts, y_test_parts = [], []
test_bank_ids = []  # remembers which bank each TEST row belongs to, for inverse-scaling later

for bank_id in bank_sequences_train:
    X_tr, y_tr = bank_sequences_train[bank_id]
    X_te, y_te = bank_sequences_test[bank_id]

    X_train_parts.append(X_tr)
    X_test_parts.append(X_te)
    y_train_parts.append(y_tr)
    y_test_parts.append(y_te)

    test_bank_ids.extend([bank_id] * len(X_te))

X_train_lstm = np.concatenate(X_train_parts).reshape(-1, sequence, 1)
X_test_lstm = np.concatenate(X_test_parts).reshape(-1, sequence, 1)

y_train_lstm = np.concatenate(y_train_parts)
y_test_lstm = np.concatenate(y_test_parts)

test_bank_ids = np.array(test_bank_ids)


In [ ]:
# The true (inverse-scaled) test values, shared by GRU / Bidirectional LSTM /
# CNN-LSTM below -- computed once here rather than per model.
actual_parts = []

for bank_id in sorted(set(test_bank_ids)):
    mask = test_bank_ids == bank_id
    scaler = bank_scalers[bank_id]
    actual_parts.append(scaler.inverse_transform(y_test_lstm[mask].reshape(-1, 1)))

actual = np.concatenate(actual_parts)


# GRU


In [ ]:
# Same per-bank sequences built above (X_train_lstm / X_test_lstm / y_train_lstm /
# y_test_lstm / test_bank_ids / bank_scalers) are reused for all three sequence
# models below, so they're compared on exactly the same data and split.
gru_model = Sequential([
    GRU(64, return_sequences=True, input_shape=(sequence, 1)),
    Dropout(0.2),
    GRU(64),
    Dropout(0.2),
    Dense(1),
])

gru_model.compile(optimizer='adam', loss='mse')

gru_history = gru_model.fit(
    X_train_lstm,
    y_train_lstm,
    epochs=30,
    batch_size=32,
    validation_data=(X_test_lstm, y_test_lstm),
    verbose=1,
)


In [ ]:
gru_pred_scaled = gru_model.predict(X_test_lstm)

gru_pred_parts = []

for bank_id in sorted(set(test_bank_ids)):
    mask = test_bank_ids == bank_id
    scaler = bank_scalers[bank_id]
    gru_pred_parts.append(scaler.inverse_transform(gru_pred_scaled[mask]))

gru_pred = np.concatenate(gru_pred_parts)


## GRU Results


In [ ]:
gru_mae = mean_absolute_error(actual, gru_pred)

gru_rmse = np.sqrt(mean_squared_error(actual, gru_pred))

gru_r2 = r2_score(actual, gru_pred)

print("MAE :", gru_mae)
print("RMSE:", gru_rmse)
print("R\u00b2  :", gru_r2)


In [ ]:
plt.figure(figsize=(15,5))

plt.plot(actual, label='Actual')

plt.plot(gru_pred, label='GRU')

plt.title("GRU Prediction")

plt.xlabel("Time")

plt.ylabel("Close Price")

plt.legend()

plt.show()


## GRU Training Loss


In [ ]:
plt.figure(figsize=(10,5))

plt.plot(gru_history.history['loss'], label='Training Loss')
plt.plot(gru_history.history['val_loss'], label='Validation Loss')

plt.title("GRU Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()


# Bidirectional LSTM


In [ ]:
bilstm_model = Sequential([
    Bidirectional(LSTM(64, return_sequences=True), input_shape=(sequence, 1)),
    Dropout(0.2),
    Bidirectional(LSTM(64)),
    Dropout(0.2),
    Dense(1),
])

bilstm_model.compile(optimizer='adam', loss='mse')

bilstm_history = bilstm_model.fit(
    X_train_lstm,
    y_train_lstm,
    epochs=30,
    batch_size=32,
    validation_data=(X_test_lstm, y_test_lstm),
    verbose=1,
)


In [ ]:
bilstm_pred_scaled = bilstm_model.predict(X_test_lstm)

bilstm_pred_parts = []

for bank_id in sorted(set(test_bank_ids)):
    mask = test_bank_ids == bank_id
    scaler = bank_scalers[bank_id]
    bilstm_pred_parts.append(scaler.inverse_transform(bilstm_pred_scaled[mask]))

bilstm_pred = np.concatenate(bilstm_pred_parts)


## Bidirectional LSTM Results


In [ ]:
bilstm_mae = mean_absolute_error(actual, bilstm_pred)

bilstm_rmse = np.sqrt(mean_squared_error(actual, bilstm_pred))

bilstm_r2 = r2_score(actual, bilstm_pred)

print("MAE :", bilstm_mae)
print("RMSE:", bilstm_rmse)
print("R\u00b2  :", bilstm_r2)


In [ ]:
plt.figure(figsize=(15,5))

plt.plot(actual, label='Actual')

plt.plot(bilstm_pred, label='Bidirectional LSTM')

plt.title("Bidirectional LSTM Prediction")

plt.xlabel("Time")

plt.ylabel("Close Price")

plt.legend()

plt.show()


## Bidirectional LSTM Training Loss


In [ ]:
plt.figure(figsize=(10,5))

plt.plot(bilstm_history.history['loss'], label='Training Loss')
plt.plot(bilstm_history.history['val_loss'], label='Validation Loss')

plt.title("Bidirectional LSTM Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()


# CNN-LSTM Hybrid


In [ ]:
# Conv1D extracts short local patterns (e.g. 3-5 day shapes) from each 60-day
# window before the LSTM models the longer sequence dependency across them.
cnn_lstm_model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(sequence, 1)),
    MaxPooling1D(pool_size=2),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(64),
    Dropout(0.2),
    Dense(1),
])

cnn_lstm_model.compile(optimizer='adam', loss='mse')

cnn_lstm_history = cnn_lstm_model.fit(
    X_train_lstm,
    y_train_lstm,
    epochs=30,
    batch_size=32,
    validation_data=(X_test_lstm, y_test_lstm),
    verbose=1,
)


In [ ]:
cnn_lstm_pred_scaled = cnn_lstm_model.predict(X_test_lstm)

cnn_lstm_pred_parts = []

for bank_id in sorted(set(test_bank_ids)):
    mask = test_bank_ids == bank_id
    scaler = bank_scalers[bank_id]
    cnn_lstm_pred_parts.append(scaler.inverse_transform(cnn_lstm_pred_scaled[mask]))

cnn_lstm_pred = np.concatenate(cnn_lstm_pred_parts)


## CNN-LSTM Hybrid Results


In [ ]:
cnn_lstm_mae = mean_absolute_error(actual, cnn_lstm_pred)

cnn_lstm_rmse = np.sqrt(mean_squared_error(actual, cnn_lstm_pred))

cnn_lstm_r2 = r2_score(actual, cnn_lstm_pred)

print("MAE :", cnn_lstm_mae)
print("RMSE:", cnn_lstm_rmse)
print("R\u00b2  :", cnn_lstm_r2)


In [ ]:
plt.figure(figsize=(15,5))

plt.plot(actual, label='Actual')

plt.plot(cnn_lstm_pred, label='CNN-LSTM Hybrid')

plt.title("CNN-LSTM Hybrid Prediction")

plt.xlabel("Time")

plt.ylabel("Close Price")

plt.legend()

plt.show()


## CNN-LSTM Hybrid Training Loss


In [ ]:
plt.figure(figsize=(10,5))

plt.plot(cnn_lstm_history.history['loss'], label='Training Loss')
plt.plot(cnn_lstm_history.history['val_loss'], label='Validation Loss')

plt.title("CNN-LSTM Hybrid Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()


# Model Comparison


In [ ]:
comparison = pd.DataFrame({

'Model':['XGBoost','GRU','Bidirectional LSTM','CNN-LSTM'],

'MAE':[xgb_mae,gru_mae,bilstm_mae,cnn_lstm_mae],

'RMSE':[xgb_rmse,gru_rmse,bilstm_rmse,cnn_lstm_rmse],

'R2 Score':[xgb_r2,gru_r2,bilstm_r2,cnn_lstm_r2]

})

comparison


In [ ]:
plt.figure(figsize=(8,5))

plt.bar(comparison['Model'], comparison['MAE'])

plt.title("MAE Comparison")

plt.ylabel("MAE")

plt.show()


In [ ]:
plt.figure(figsize=(8,5))

plt.bar(comparison['Model'], comparison['RMSE'])

plt.title("RMSE Comparison")

plt.ylabel("RMSE")

plt.show()


In [ ]:
plt.figure(figsize=(8,5))

plt.bar(comparison['Model'], comparison['R2 Score'])

plt.title("R\u00b2 Score Comparison")

plt.ylabel("R\u00b2")

plt.show()


# Save Models


In [ ]:
joblib.dump(xgb_model, 'XGBoost.pkl')


In [ ]:
gru_model.save("GRU.keras")


In [ ]:
bilstm_model.save("BiLSTM.keras")


In [ ]:
cnn_lstm_model.save("CNN_LSTM.keras")


Conclusion

Four additional models were implemented for bank stock price forecasting on the
same cleaned, leakage-safe dataset used for ARIMA / Random Forest / LSTM:
XGBoost, GRU, Bidirectional LSTM, and a CNN-LSTM hybrid.

Performance was evaluated using MAE, RMSE, and R\u00b2 Score, consistent with the
original notebook, so results can be compared directly across all seven models.

XGBoost extends the tree-based comparison started by Random Forest, generally
training faster and often matching or beating it on tabular lag/moving-average
features. GRU is a cheaper alternative to LSTM with fewer parameters per cell;
Bidirectional LSTM reads each 60-day window in both directions; and the CNN-LSTM
hybrid extracts short local patterns with a convolutional layer before modelling
the longer sequence dependency with LSTM.
